In [103]:
import sys
sys.path.append('../..')

In [104]:
import torch
from xaikd import datasets

In [41]:
artifact = torch.load("/home/pat/projects/playground/tmp", map_location=torch.device('cpu'))

In [42]:
artifact.keys()

dict_keys(['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers'])

In [44]:
state_dict = artifact["state_dict"]

In [45]:
backbone_keys = list(filter(lambda k: "backbone" in k, state_dict.keys()))

In [46]:
from torchvision.models import resnet

In [84]:
from torch import nn 

def _resnet18_cifar(num_classes: int=10) -> nn.Module:
    model = resnet.resnet18(weights=None)

    # why we use this? (ask Florian?)
    model.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    model.maxpool = nn.Identity()

    model.avgpool = nn.AvgPool2d(kernel_size=4)
    model.fc = nn.Linear(512, num_classes)

    model.num_classes = num_classes

    return model

model = _resnet18_cifar()


first_level_names = list(map(lambda x: x[0], list(model.named_children())))
first_level_names = list(filter(lambda n: (not "relu" in n) and (not "maxpool" in n), first_level_names))

In [85]:
first_level_names

['conv1', 'bn1', 'layer1', 'layer2', 'layer3', 'layer4', 'avgpool', 'fc']

In [86]:
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): Identity()
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), p

In [131]:
def mapstate_dict(seed=1, skip_loading=False):
    
    _m = _resnet18_cifar()
    
    
    
    key_group = dict()
    
    for v in backbone_keys:
        if skip_loading:
            print("We skip loading")
            break

        k = v.replace("backbone.", "")
        k = k.split(".")
        item = k[-1]
        group = tuple(k[:-1])
        
        value = state_dict[v]
        
        if group not in key_group:
            key_group[group] = dict()
        
        key_group[group][item] = value

            
    for group in key_group.keys():
        attr = first_level_names[int(group[0])]
        print(f"[group={group}] (attr={attr})")
        if len(group) == 1:
#             print(f"load statedict to {group}")
#             print("with ", key_group[group])
            module = getattr(_m, attr)
            
        elif len(group) == 3:
#             print(f"[{group}] do nothing for now")
            second_level = int(group[1])
            third_level = group[2]
            
            parent_module = getattr(_m, attr)[second_level]
            module = getattr(parent_module, third_level) 
        elif len(group) == 4:
            second_level = int(group[1])
            third_level = group[2]
            fouth_level = int(group[3])
            
            assert third_level == "shortcut"
            
            parent_module = getattr(_m, attr)[second_level]
            parent_module = getattr(parent_module, "downsample")

            module = parent_module[fouth_level]
        
        module.load_state_dict(key_group[group])
        print("> Ok!")
        
    _m.eval()
        
    return _m
        
loaded_model = mapstate_dict()

[group=('0',)] (attr=conv1)
> Ok!
[group=('1',)] (attr=bn1)
> Ok!
[group=('2', '0', 'conv1')] (attr=layer1)
> Ok!
[group=('2', '0', 'bn1')] (attr=layer1)
> Ok!
[group=('2', '0', 'conv2')] (attr=layer1)
> Ok!
[group=('2', '0', 'bn2')] (attr=layer1)
> Ok!
[group=('2', '1', 'conv1')] (attr=layer1)
> Ok!
[group=('2', '1', 'bn1')] (attr=layer1)
> Ok!
[group=('2', '1', 'conv2')] (attr=layer1)
> Ok!
[group=('2', '1', 'bn2')] (attr=layer1)
> Ok!
[group=('3', '0', 'conv1')] (attr=layer2)
> Ok!
[group=('3', '0', 'bn1')] (attr=layer2)
> Ok!
[group=('3', '0', 'conv2')] (attr=layer2)
> Ok!
[group=('3', '0', 'bn2')] (attr=layer2)
> Ok!
[group=('3', '0', 'shortcut', '0')] (attr=layer2)
> Ok!
[group=('3', '0', 'shortcut', '1')] (attr=layer2)
> Ok!
[group=('3', '1', 'conv1')] (attr=layer2)
> Ok!
[group=('3', '1', 'bn1')] (attr=layer2)
> Ok!
[group=('3', '1', 'conv2')] (attr=layer2)
> Ok!
[group=('3', '1', 'bn2')] (attr=layer2)
> Ok!
[group=('4', '0', 'conv1')] (attr=layer3)
> Ok!
[group=('4', '0', 'bn1

# Sanity Check

In [146]:
from pytorch_lightning import LightningModule, Trainer
from torch.utils.data import DataLoader
import torch
from torch import Tensor
from torch.nn import functional as F
from copy import deepcopy
import torch.distributed as dist

from tqdm import tqdm

# code for kNN prediction from here:
# https://colab.research.google.com/github/facebookresearch/moco/blob/colab-notebook/colab/moco_cifar10_demo.ipynb


def knn_predict(
    feature: Tensor,
    feature_bank: Tensor,
    feature_labels: Tensor,
    num_classes: int,
    knn_k: int = 200,
    knn_t: float = 0.1,
) -> Tensor:
    """Run kNN predictions on features based on a feature bank

    This method is commonly used to monitor performance of self-supervised
    learning methods.

    The default parameters are the ones
    used in https://arxiv.org/pdf/1805.01978v1.pdf.

    Args:
        feature:
            Tensor with shape (B, D) for which you want predictions.
        feature_bank:
            Tensor of shape (D, N) of a database of features used for kNN.
        feature_labels:
            Labels with shape (N,) for the features in the feature_bank.
        num_classes:
            Number of classes (e.g. `10` for CIFAR-10).
        knn_k:
            Number of k neighbors used for kNN.
        knn_t:
            Temperature parameter to reweights similarities for kNN.

    Returns:
        A tensor containing the kNN predictions

    Examples:
        >>> images, targets, _ = batch
        >>> feature = backbone(images).squeeze()
        >>> # we recommend to normalize the features
        >>> feature = F.normalize(feature, dim=1)
        >>> pred_labels = knn_predict(
        >>>     feature,
        >>>     feature_bank,
        >>>     targets_bank,
        >>>     num_classes=10,
        >>> )
    """
    # compute cos similarity between each feature vector and feature bank ---> (B, N)
    sim_matrix = torch.mm(feature, feature_bank)
    # (B, K)
    sim_weight, sim_indices = sim_matrix.topk(k=knn_k, dim=-1)
    # (B, K)
    sim_labels = torch.gather(
        feature_labels.expand(feature.size(0), -1), dim=-1, index=sim_indices
    )
    # we do a reweighting of the similarities
    sim_weight = (sim_weight / knn_t).exp()
    # counts for each class
    one_hot_label = torch.zeros(
        feature.size(0) * knn_k, num_classes, device=sim_labels.device
    )
    # (B*K, C)
    one_hot_label = one_hot_label.scatter(
        dim=-1, index=sim_labels.view(-1, 1), value=1.0
    )
    # weighted score ---> (B, C)
    pred_scores = torch.sum(
        one_hot_label.view(feature.size(0), -1, num_classes)
        * sim_weight.unsqueeze(dim=-1),
        dim=1,
    )
    pred_labels = pred_scores.argsort(dim=-1, descending=True)
    return pred_labels

class BenchmarkModule(LightningModule):
    """A PyTorch Lightning Module for automated kNN callback

    At the end of every training epoch we create a feature bank by feeding the
    `dataloader_kNN` passed to the module through the backbone.
    At every validation step we predict features on the validation data.
    After all predictions on validation data (validation_epoch_end) we evaluate
    the predictions on a kNN classifier on the validation data using the
    feature_bank features from the train data.

    We can access the highest test accuracy during a kNN prediction
    using the `max_accuracy` attribute.

    Attributes:
        backbone:
            The backbone model used for kNN validation. Make sure that you set the
            backbone when inheriting from `BenchmarkModule`.
        max_accuracy:
            Floating point number between 0.0 and 1.0 representing the maximum
            test accuracy the benchmarked model has achieved.
        dataloader_kNN:
            Dataloader to be used after each training epoch to create feature bank.
        num_classes:
            Number of classes. E.g. for cifar10 we have 10 classes. (default: 10)
        knn_k:
            Number of nearest neighbors for kNN
        knn_t:
            Temperature parameter for kNN

    Examples:
        >>> class SimSiamModel(BenchmarkingModule):
        >>>     def __init__(dataloader_kNN, num_classes):
        >>>         super().__init__(dataloader_kNN, num_classes)
        >>>         resnet = lightly.models.ResNetGenerator('resnet-18')
        >>>         self.backbone = nn.Sequential(
        >>>             *list(resnet.children())[:-1],
        >>>             nn.AdaptiveAvgPool2d(1),
        >>>         )
        >>>         self.resnet_simsiam =
        >>>             lightly.models.SimSiam(self.backbone, num_ftrs=512)
        >>>         self.criterion = lightly.loss.SymNegCosineSimilarityLoss()
        >>>
        >>>     def forward(self, x):
        >>>         self.resnet_simsiam(x)
        >>>
        >>>     def training_step(self, batch, batch_idx):
        >>>         (x0, x1), _, _ = batch
        >>>         x0, x1 = self.resnet_simsiam(x0, x1)
        >>>         loss = self.criterion(x0, x1)
        >>>         return loss
        >>>     def configure_optimizers(self):
        >>>         optim = torch.optim.SGD(
        >>>             self.resnet_simsiam.parameters(), lr=6e-2, momentum=0.9
        >>>         )
        >>>         return [optim]
        >>>
        >>> model = SimSiamModel(dataloader_train_kNN)
        >>> trainer = pl.Trainer()
        >>> trainer.fit(
        >>>     model,
        >>>     train_dataloader=dataloader_train_ssl,
        >>>     val_dataloaders=dataloader_test
        >>> )
        >>> # you can get the peak accuracy using
        >>> print(model.max_accuracy)

    """

    def __init__(
        self,
        model,
        dataloader_kNN: DataLoader,
        num_classes: int,
        knn_k: int = 200,
        knn_t: float = 0.1,
    ):
        super().__init__()
        
        model = deepcopy(model)
        model.fc = nn.Identity()
        
        self.backbone = model
        self.max_accuracy = 0.0
        self.dataloader_kNN = dataloader_kNN
        self.num_classes = num_classes
        self.knn_k = knn_k
        self.knn_t = knn_t

        self._train_features: Optional[Tensor] = None
        self._train_targets: Optional[Tensor] = None
        self._val_predicted_labels: List[Tensor] = []
        self._val_targets: List[Tensor] = []

    def on_validation_epoch_start(self) -> None:
        train_features = []
        train_targets = []
        with torch.no_grad():
            for data in tqdm(self.dataloader_kNN):
                img, target = data
                img = img.to(self.device)
                target = target.to(self.device)
                feature = self.backbone(img).squeeze()
                feature = F.normalize(feature, dim=1)
                if (
                    dist.is_available()
                    and dist.is_initialized()
                    and dist.get_world_size() > 0
                ):
                    # gather features and targets from all processes
                    feature = torch.cat(dist.gather(feature), 0)
                    target = torch.cat(dist.gather(target), 0)
                train_features.append(feature)
                train_targets.append(target)

        self._train_features = torch.cat(train_features, dim=0).t().contiguous()
        self._train_targets = torch.cat(train_targets, dim=0).t().contiguous()

    def validation_step(self, batch, batch_idx) -> None:
        # we can only do kNN predictions once we have a feature bank
        if self._train_features is not None and self._train_targets is not None:
            images, targets = batch
            feature = self.backbone(images).squeeze()
            feature = F.normalize(feature, dim=1)
            predicted_labels = knn_predict(
                feature,
                self._train_features,
                self._train_targets,
                self.num_classes,
                self.knn_k,
                self.knn_t,
            )
            if dist.is_initialized() and dist.get_world_size() > 0:
                # gather predictions and targets from all processes
                predicted_labels = torch.cat(dist.gather(predicted_labels), 0)
                targets = torch.cat(dist.gather(targets), 0)

            self._val_predicted_labels.append(predicted_labels.cpu())
            self._val_targets.append(targets.cpu())

    def on_validation_epoch_end(self) -> None:
        if self._val_predicted_labels and self._val_targets:
            predicted_labels = torch.cat(self._val_predicted_labels, dim=0)
            targets = torch.cat(self._val_targets, dim=0)
            top1 = (predicted_labels[:, 0] == targets).float().sum()
            acc = top1 / len(targets)
            if acc > self.max_accuracy:
                self.max_accuracy = acc.item()
            self.log("kNN_accuracy", acc * 100.0, prog_bar=True)
            print(f"acc={acc}")
        self._val_predicted_labels.clear()
        self._val_targets.clear()

In [147]:
dataset = datasets.construct("cifar10")
train_loader = dataset.loader(train_split=True)
val_loader = dataset.loader(train_split=False)

In [150]:
trainer = Trainer()

trainer.validate(
    BenchmarkModule(
        mapstate_dict(skip_loading=False),
        train_loader,
        num_classes=10
    ),
    dataloaders=val_loader
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


[group=('0',)] (attr=conv1)
> Ok!
[group=('1',)] (attr=bn1)
> Ok!
[group=('2', '0', 'conv1')] (attr=layer1)
> Ok!
[group=('2', '0', 'bn1')] (attr=layer1)
> Ok!
[group=('2', '0', 'conv2')] (attr=layer1)
> Ok!
[group=('2', '0', 'bn2')] (attr=layer1)
> Ok!
[group=('2', '1', 'conv1')] (attr=layer1)
> Ok!
[group=('2', '1', 'bn1')] (attr=layer1)
> Ok!
[group=('2', '1', 'conv2')] (attr=layer1)
> Ok!
[group=('2', '1', 'bn2')] (attr=layer1)
> Ok!
[group=('3', '0', 'conv1')] (attr=layer2)
> Ok!
[group=('3', '0', 'bn1')] (attr=layer2)
> Ok!
[group=('3', '0', 'conv2')] (attr=layer2)
> Ok!
[group=('3', '0', 'bn2')] (attr=layer2)
> Ok!
[group=('3', '0', 'shortcut', '0')] (attr=layer2)
> Ok!
[group=('3', '0', 'shortcut', '1')] (attr=layer2)
> Ok!
[group=('3', '1', 'conv1')] (attr=layer2)
> Ok!
[group=('3', '1', 'bn1')] (attr=layer2)
> Ok!
[group=('3', '1', 'conv2')] (attr=layer2)
> Ok!
[group=('3', '1', 'bn2')] (attr=layer2)
> Ok!
[group=('4', '0', 'conv1')] (attr=layer3)
> Ok!
[group=('4', '0', 'bn1

Validation: 0it [00:00, ?it/s]


  5%|████████                                                                                                                                                                  | 37/782 [00:18<07:34,  1.64it/s]


 10%|████████████████▎                                                                                                                                                         | 75/782 [00:41<06:58,  1.69it/s]


...

 92%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊             | 721/782 [06:33<00:26,  2.27it/s]


 97%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████     | 759/782 [06:54<00:17,  1.35it/s]


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [07:05<00:00,  1.84it/s]


acc=0.8738999962806702
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Runningstage.validating metric      DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      kNN_accuracy           87.38999938964844
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'kNN_accuracy': 87.38999938964844}]

- with load_state_dict: 87.38999938964844
- w/o load_state_dict:  37.900001525878906


In [152]:
loaded_model = mapstate_dict(skip_loading=False);

[group=('0',)] (attr=conv1)
> Ok!
[group=('1',)] (attr=bn1)
> Ok!
[group=('2', '0', 'conv1')] (attr=layer1)
> Ok!
[group=('2', '0', 'bn1')] (attr=layer1)
> Ok!
[group=('2', '0', 'conv2')] (attr=layer1)
> Ok!
[group=('2', '0', 'bn2')] (attr=layer1)
> Ok!
[group=('2', '1', 'conv1')] (attr=layer1)
> Ok!
[group=('2', '1', 'bn1')] (attr=layer1)
> Ok!
[group=('2', '1', 'conv2')] (attr=layer1)
> Ok!
[group=('2', '1', 'bn2')] (attr=layer1)
> Ok!
[group=('3', '0', 'conv1')] (attr=layer2)
> Ok!
[group=('3', '0', 'bn1')] (attr=layer2)
> Ok!
[group=('3', '0', 'conv2')] (attr=layer2)
> Ok!
[group=('3', '0', 'bn2')] (attr=layer2)
> Ok!
[group=('3', '0', 'shortcut', '0')] (attr=layer2)
> Ok!
[group=('3', '0', 'shortcut', '1')] (attr=layer2)
> Ok!
[group=('3', '1', 'conv1')] (attr=layer2)
> Ok!
[group=('3', '1', 'bn1')] (attr=layer2)
> Ok!
[group=('3', '1', 'conv2')] (attr=layer2)
> Ok!
[group=('3', '1', 'bn2')] (attr=layer2)
> Ok!
[group=('4', '0', 'conv1')] (attr=layer3)
> Ok!
[group=('4', '0', 'bn1

In [154]:
torch.save(loaded_model.state_dict(), "resnet18simclr.pth")